In [ ]:
#img input

import cv2
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Load Image & Pre-process ---
img_path = r"C:\Users\rishi\Desktop\SURF\CODING\Image Processing\Image Detection\Test1.jpg" 
frame = cv2.imread(img_path)
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

img_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(img_gray, (13, 13), 0)

# --- 2. Total Carbon Black Detection ---
ret, mask = cv2.threshold(blur, 140, 255, cv2.THRESH_BINARY_INV)
mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))

# ==========================================
# --- NEW: THE PARTICLE FILTER (Area > 50) ---
# ==========================================
raw_cb_contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Create a blank canvas to hold ONLY the valid carbon black particles
filtered_cb_mask = np.zeros_like(mask_clean)

# Draw only the particles larger than 50 pixels
for cnt in raw_cb_contours:
    if cv2.contourArea(cnt) > 50:
        cv2.drawContours(filtered_cb_mask, [cnt], -1, 255, -1)

# --- 3. Bubble Detection (Canny Subtraction) ---
edges = cv2.Canny(mask, 5, 15)
edges_normal = cv2.Canny(blur, 5, 15)

difference = cv2.absdiff(edges, edges_normal)
closed_diff = cv2.morphologyEx(difference, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
diff_contours, _ = cv2.findContours(closed_diff, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Draw the solid bubble zones
bubble_zone_mask = np.zeros_like(mask)
for cnt in diff_contours:
    if cv2.contourArea(cnt) > 690:  # Your tuned bubble area threshold
        cv2.drawContours(bubble_zone_mask, [cnt], -1, 255, -1)

# ==========================================
# --- 4. THE CATEGORIZATION ---
# ==========================================

# Category A: CB WITH Bubbles (Intersection)
# Notice we are now using 'filtered_cb_mask' instead of 'mask_clean'
cb_with_bubbles = cv2.bitwise_and(filtered_cb_mask, bubble_zone_mask)

# Category B: CB WITHOUT Bubbles (Total Valid CB minus Bubbled CB)
cb_without_bubbles = cv2.bitwise_xor(filtered_cb_mask, cb_with_bubbles)

# ==========================================
# --- 5. Extract Contours & Visualize ---
# ==========================================

contours_with, _ = cv2.findContours(cb_with_bubbles, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours_without, _ = cv2.findContours(cb_without_bubbles, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Create blank canvases to draw our results on
canvas_with = np.zeros_like(frame_rgb)
canvas_without = np.zeros_like(frame_rgb)
master_overlay = frame_rgb.copy()

# Draw "With Bubbles" as Green
cv2.drawContours(canvas_with, contours_with, -1, (0, 255, 0), 2)
cv2.drawContours(master_overlay, contours_with, -1, (0, 255, 0), 2)

# Draw "Without Bubbles" as Red
cv2.drawContours(canvas_without, contours_without, -1, (255, 0, 0), 2)
cv2.drawContours(master_overlay, contours_without, -1, (255, 0, 0), 2)

# Display the Final Dashboards
print(f"Active Particles (With Bubbles): {len(contours_with)}")
print(f"Dormant Particles (No Bubbles): {len(contours_without)}")

'''plt.figure(figsize=(18, 6))

plt.subplot(1, 4, 1)
plt.imshow(img_gray, cmap='gray')
plt.title("Original Grayscale Image")
plt.axis('off')

plt.subplot(1,4, 2)
plt.imshow(canvas_with)
plt.title(f"CB WITH Bubbles ({len(contours_with)} found)")
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(canvas_without)
plt.title(f"CB WITHOUT Bubbles ({len(contours_without)} found)")
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(master_overlay)
plt.title("Master Overlay (Green = Active, Red = Dormant)")
plt.axis('off')

plt.tight_layout()
#plt.show()

'''
plt.figure(figsize=(20, 12))
plt.subplot(1,2,1)
plt.imshow(img_gray, cmap='gray')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(master_overlay)
plt.axis('off')

plt.show()

In [ ]:
# vid input

import cv2
import numpy as np

# --- 1. Setup Video / Live Feed ---
vid_path = r"C:\Users\rishi\Desktop\SURF\CODING\Image Processing\CB_CTAC_06082026\20x_2mw_001.mp4" 
cap = cv2.VideoCapture(vid_path)

# ==========================================
# --- NEW: SETUP VIDEO WRITER ---
# ==========================================
# Get the width, height, and Frames Per Second (FPS) of the original video
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps == 0: fps = 30 # Fallback just in case

# Define the output path and codec (mp4v is standard for .mp4 files)
#out_path = r"C:\Users\rishi\Desktop\SURF\CODING\Image Processing\CB_CTAC_06082026\Output_Tracked.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
#out_video = cv2.VideoWriter(out_path, fourcc, fps, (frame_width, frame_height))
# ==========================================

# Initialize our master arrays
time_series_with = []
time_series_without = []

# Your tuned area thresholds
cb_area_threshold = 50
bubble_area_threshold = 690

print("Starting live particle extraction and video export...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break # Video ended or feed interrupted
        
    current_time_ms = cap.get(cv2.CAP_PROP_POS_MSEC)

    # --- 2. Pre-process ---
    img_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(img_gray, (9, 9), 0)

    # --- 3. Total Carbon Black Detection & Area Filter ---
    ret, mask = cv2.threshold(blur, 140, 255, cv2.THRESH_BINARY_INV)
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))

    raw_cb_contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filtered_cb_mask = np.zeros_like(mask_clean)
    
    for cnt in raw_cb_contours:
        if cv2.contourArea(cnt) > cb_area_threshold:
            cv2.drawContours(filtered_cb_mask, [cnt], -1, 255, -1)

    # --- 4. Bubble Detection (Canny Subtraction) ---
    edges = cv2.Canny(mask, 5, 15)
    edges_normal = cv2.Canny(blur, 5, 15)

    difference = cv2.absdiff(edges, edges_normal)
    closed_diff = cv2.morphologyEx(difference, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    diff_contours, _ = cv2.findContours(closed_diff, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    bubble_zone_mask = np.zeros_like(mask)
    for cnt in diff_contours:
        if cv2.contourArea(cnt) > bubble_area_threshold:
            cv2.drawContours(bubble_zone_mask, [cnt], -1, 255, -1)

    # --- 5. THE CATEGORIZATION ---
    cb_with_bubbles = cv2.bitwise_and(filtered_cb_mask, bubble_zone_mask)
    cb_without_bubbles = cv2.bitwise_xor(filtered_cb_mask, cb_with_bubbles)

    # --- 6. Extract Coordinates, Draw & Save ---
    contours_with, _ = cv2.findContours(cb_with_bubbles, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours_without, _ = cv2.findContours(cb_without_bubbles, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Process "Active" Particles (With Bubbles) -> DRAW GREEN
    for cnt in contours_with:
        (x, y), radius = cv2.minEnclosingCircle(cnt)
        time_series_with.append([current_time_ms, float(x), float(y), float(radius)])
        # Draw a Green targeting circle on the original frame
        cv2.circle(frame, (int(x), int(y)), int(radius), (0, 255, 0), 2)
        cv2.circle(frame, (int(x), int(y)), 2, (0, 255, 0), -1)

    # Process "Dormant" Particles (Without Bubbles) -> DRAW RED
    for cnt in contours_without:
        (x, y), radius = cv2.minEnclosingCircle(cnt)
        time_series_without.append([current_time_ms, float(x), float(y), float(radius)])
        # Draw a Red targeting circle on the original frame (OpenCV uses BGR, so Red is 0,0,255)
        cv2.circle(frame, (int(x), int(y)), int(radius), (0, 0, 255), 2)
        cv2.circle(frame, (int(x), int(y)), 2, (0, 0, 255), -1)

    # Write the beautifully annotated frame to your output MP4 file!
    #out_video.write(frame)

    # Show the annotated video live on your screen
    cv2.imshow("Live Tracking Output", frame) 
    if cv2.waitKey(1) & 0xFF == ord('q'): break

# Clean up and officially save the video file
cap.release()
#out_video.release() # CRITICAL: This is what physically saves the file to your hard drive
cv2.destroyAllWindows()

# Convert the massive lists into powerful NumPy arrays
array_with_time = np.array(time_series_with)
array_without_time = np.array(time_series_without)

print("\n--- Processing Complete ---")
#print(f"Video saved to: {out_path}")
print(f"Total 'Active' Datapoints Collected: {len(array_with_time)}")
print(f"Total 'Dormant' Datapoints Collected: {len(array_without_time)}")